# Coders of Delhi Project

## Task 1: Load the User Data

In this task, we load the dataset containing information about users, their friends, and liked pages. The data is in JSON format.

In [1]:
import json
def load_data(filename):
    with open(filename,"r") as f:
        data=json.load(f)
    return data    

In [2]:
data=load_data("data.json")


In [3]:
data


{'users': [{'id': 1, 'name': 'Amit', 'friends': [2, 3], 'liked_pages': [101]},
  {'id': 2, 'name': 'Priya', 'friends': [1, 4], 'liked_pages': [102]},
  {'id': 3, 'name': 'Rahul', 'friends': [1], 'liked_pages': [101, 103]},
  {'id': 4, 'name': 'Sara', 'friends': [2], 'liked_pages': [104]}],
 'pages': [{'id': 101, 'name': 'Python Developers'},
  {'id': 102, 'name': 'Data Science Enthusiasts'},
  {'id': 103, 'name': 'AI & ML Community'},
  {'id': 104, 'name': 'Web Dev Hub'}]}

In [11]:
type(data)

dict

## Task 2: Read and Display the Data using Python

In this task, we read the JSON dataset using Python and display the information in a structured format. The goal is to understand user details, their connections (friends), and the pages they like.

### Steps Performed:
- Loaded the JSON data from file
- Displayed user details and their friends
- Displayed liked pages for each user
- Printed all available pages

### Output:
The data was successfully read and displayed, showing user connections and page preferences.

In [18]:
#function to display users and their connections
def display_users(data):
    print("\nUsers and their Connections:\n")
    for user in data['users']:
        print(f"{user['name']}(Id:{user['id']}-friends{user['friends']}-liked_pages{user['liked_pages']}")
    print("\nPages:\n")  
    for page in data['pages']  :
        print(f"{page['id']}:{page['name']}")
display_users(data)                



Users and their Connections:

Amit(Id:1-friends[2, 3]-liked_pages[101]
Priya(Id:2-friends[1, 4]-liked_pages[102]
Sara(Id:4-friends[2]-liked_pages[104]

Pages:

101:Python Developers
102:Data Science Enthusiasts
103:AI & ML Community
104:Web Development


## Task 3: Identify Issues in the Data

In this task, we analyze the dataset to identify missing, incorrect, and inconsistent values.

### Issues Found:
- User with ID 3 has an empty name
- User with ID 4 has duplicate friend entries
- User with ID 5 has no connections or liked pages (inactive user)
- Duplicate page IDs are present in the pages list

### Conclusion:
The dataset contains several inconsistencies and missing values that need to be cleaned before further processing and analysis.

In [24]:
#loading data
data=load_data("data2.json")
data

{'users': [{'id': 1, 'name': 'Amit', 'friends': [2, 3], 'liked_pages': [101]},
  {'id': 2, 'name': 'Priya', 'friends': [1, 4], 'liked_pages': [102]},
  {'id': 3, 'name': '', 'friends': [1], 'liked_pages': [101, 103]},
  {'id': 4, 'name': 'Sara', 'friends': [2, 2], 'liked_pages': [104]},
  {'id': 5, 'name': 'Amit', 'friends': [], 'liked_pages': []}],
 'pages': [{'id': 101, 'name': 'Python Developers'},
  {'id': 102, 'name': 'Data Science Enthusiasts'},
  {'id': 103, 'name': 'AI & ML Community'},
  {'id': 104, 'name': 'Web Dev Hub'},
  {'id': 104, 'name': 'Web Development'}]}

## Task 4: Clean the Data

In this task, we clean the dataset by removing incorrect, missing, and duplicate entries identified earlier.

### Cleaning Steps:
1. Remove users with missing names.
2. Remove duplicate friend entries.
3. Remove inactive users (users with no friends and no liked pages).
4. Remove duplicate page IDs from the pages list.

### Result:
The dataset is now cleaned and structured, making it ready for further analysis and recommendations.

In [29]:
def clean_data(data):
    #Remove user with empty name
    cleaned_users=[]
    for user in data['users']:
        if user['name'].strip()!="":
            cleaned_users.append(user)
    data['users']=cleaned_users
    #remove duplicate friends
    for user in data['users']:
        user['friends']=list(set(user['friends']))
    #remove inactive users
    active_user=[]
    for user in data['users']:
        if user['friends']or user['liked_pages']:
            active_user.append(user)
    data['users']=active_user  
    #Remove dupliacte pages
    unique_pages={}
    for page in data['pages']:
        unique_pages[page['id']]=page
        data['pages']=list(unique_pages.values())
    return data        
cleaned_data=clean_data(data)


with open("clean_data.json","w")as f:
    json.dump(cleaned_data,f,indent=4)
print("Data has been cleaned successfully")    
          
            
        

    
    


Data has been cleaned successfully


## Task 5: Understand the Logic

### How "People You May Know" Works:

If User A and User B are not friends but have **mutual friends**, we suggest User B to User A.

- More mutual friends = higher priority recommendation

### Example:

- Amit (ID: 1) is friends with Priya (ID: 2) and Rahul (ID: 3)  
- Priya (ID: 2) is friends with Sara (ID: 4)  

- Amit is not directly friends with Sara, but they share **Priya as a mutual friend**

👉 So, we suggest **Sara to Amit** as "People You May Know"

### Note:

There can be multiple suggestions.  
Users with more mutual friends should be ranked higher, as they are more likely to know each other.

---


## Task 6: Implement the Algorithm

In this task, we will implement the "People You May Know" feature using Python.

### Steps:

1. Find all friends of a given user.  
2. Identify mutual friends between users who are not directly connected.  
3. Rank recommendations based on the number of mutual friends.  

### Goal:

The goal is to suggest new connections to users based on shared mutual friends.  
Users with more mutual connections will be ranked higher in the recommendations.

In [33]:
data=load_data("clean_data.json")
def people_you_may_know(user_id, data):
    suggestions = {}

    # Step 1: find current user
    for user in data['users']:
        if user['id'] == user_id:
            friends = user['friends']
            break

    # Step 2: loop through friends
    for f in friends:
        for u in data['users']:
            if u['id'] == f:

                # Step 3: friends of friend
                for fof in u['friends']:

                    # Step 4: filter
                    if fof != user_id and fof not in friends:
                        
                        # Step 5: count mutual friends
                        if fof not in suggestions:
                            suggestions[fof] = 0
                        else:
                            suggestions[fof] += 1

    # Step 6: sort by mutual friends (ranking)
    sorted_suggestions = sorted(suggestions.items(), key=lambda x: x[1], reverse=True)

    return sorted_suggestions
user_id=1    
result=people_you_may_know(user_id,data)    
print(f"People you may know for User{user_id}:{result}")



People you may know for User1:[(4, 0)]


## Task 7: Understanding the Recommendation Logic

### How "Pages You Might Like" Works:

- Users engage with pages (like, comment, share, etc.).
- If two users have interacted with similar pages, they are likely to have common interests.
- For this implementation, we consider liking a page as an interaction.
- Pages followed by similar users should be recommended.

### Example(in massive_data.json):

- Amit (ID: 1) likes:
  - Python Hub (Page ID: 101)
  - AI World (Page ID: 102)

- Priya (ID: 2) likes:
  - AI World (Page ID: 102)
  - Data Science Daily (Page ID: 103)

- Since Amit and Priya both like **AI World (102)**:
  - Recommend **Data Science Daily (103)** to Amit
  - Recommend **Python Hub (101)** to Priya

---

### Concept Used: Collaborative Filtering

> "If two people like the same thing, they might like other things each one likes too."

This is a basic form of a real-world recommendation system.

## Task 8: Implement the Algorithm

In this task, we will implement a recommendation system for **"Pages You Might Like"**.

### Steps:

1. Map users to the pages they have interacted with (liked pages).  
2. Identify pages liked by users who have similar interests.  
3. Rank recommendations based on common interactions (shared likes).  

### Goal:

The goal is to recommend new pages to users based on the pages liked by other users with similar interests.  
Pages with more common interactions will be ranked higher in the recommendations.

In [36]:
data=load_data("massive_data.json")

In [38]:
def pages_you_might_like(user_id,data):
    #dictionary to save user interactions with pages
    information={}
    for user in data['users']:
        user_id=user['id']
        liked_pages=user['liked_pages']
        information[user_id]=set(liked_pages)
        if user_id not in information:
            return []
        user_liked_pages=information[user_id]    
        page_suggestions={}
        for other_user,pages in information.items():
            if other_user != user_id:
                shared_pages=user_liked_pages.intersection(pages)
            for page in pages :
                if page not in user_liked_pages:
                    page_suggestions[page]=page_suggestions.get(page,0)+len(shared_pages) 
            # Sort recommended pages based on the number of shared interactions
    sorted_pages = sorted(page_suggestions.items(), key=lambda x: x[1], reverse=True)
    return [page_id for page_id, _ in sorted_pages]
 
# Load data
data = load_data("clean_data.json")
user_id = 1  # Example: Finding recommendations for Amit
page_recommendations = pages_you_might_like(user_id, data)
print(f"Pages You Might Like for User {user_id}: {page_recommendations}")        
           







Pages You Might Like for User 1: [101, 102]
